# A multimodal MRI fingerprint

This chapter reproduces Figure 10 from (Cottaar et al. 2026). The
original code can be found in the
[gitlab](https://git.fmrib.ox.ac.uk/ndcn0236/mcmr_paper_figures/-/blob/main/Figure_10/MultiModal.ipynb?ref_type=heads).
We combine four measurements into a fingerprint of the simulated tissue:
diffusion weighting, spin echo, gradient echo, and magnetisation
transfer. We then change the axon geometry and observe which fingerprint
components respond.

## 1. Define the acquisition modalities

The diffusion-weighted spin echo and its b=0 reference form a normalized
diffusion signal. The spin echo, gradient echo, and
magnetisation-transfer sequence add complementary contrasts.

In [1]:
using Random
using MCMRSimulator
using MRIBuilder
using CairoMakie

Random.seed!(2)
set_theme!(Theme(fontsize=20))

dwi = MRIBuilder.DiffusionSpinEcho(bval=1.0, scanner=MRIBuilder.Siemens_Prisma)
dwi_b0 = MRIBuilder.adjust(dwi, diffusion=(scale=0.0,))
spin_echo = MRIBuilder.SpinEcho(TE=40, scanner=MRIBuilder.Siemens_Prisma)
gradient_echo = MRIBuilder.GradientEcho(TE=40, scanner=MRIBuilder.Siemens_Prisma)

no_mt_sequences = [dwi, dwi_b0, spin_echo, gradient_echo]
mt_sequence = MagnetisationTransfer(
    saturation=(
        nrepeat=10,
        pulse=(type=SincPulse, duration=10.0, frequency=1.5, flip_angle=700),
        interblock_delay=100,
    ),
    scanner=MRIBuilder.Siemens_Prisma
)

MRIBuilder.plot_sequence(dwi)
MRIBuilder.plot_sequence(spin_echo)
MRIBuilder.plot_sequence(gradient_echo)


******************************************************************************
This program contains Ipopt, a library for large-scale nonlinear optimization.
 Ipopt is released as open source code under the Eclipse Public License (EPL).
         For more information visit https://github.com/coin-or/Ipopt
******************************************************************************


<img width=600 height=450 style='object-fit: contain; height: auto;' src="data:image/png;base64, iVBORw0KGgoAAAANSUhEUgAAAlgAAAHCCAYAAAAzc7dkAAAAAXNSR0IArs4c6QAAAARnQU1BAACxjwv8YQUAAAAgY0hSTQAAeiYAAICEAAD6AAAAgOgAAHUwAADqYAAAOpgAABdwnLpRPAAAAAlwSFlzAAAOxAAADsQBlSsOGwAAIABJREFUeAHtwQu8D/Th+P/XOedtOudI7gcnMffLcYncfqwRYo3aLHRRiMWmWJuKZi6rZLLqW6HSkiMiWdO2poXk2pJuZN+RJlMd0Yk2cj3n9/88Hr/zf5zHeaDbu52D1/MZ8v8/SJIkKZqAJEmSogpIkiQpqoAkSZKiCkiSJCmqgCRJkqIKSJIkKaqAJEmSogpIkiQpqoAkSZKiCkiSJCmqgCRJkqIKSJIkKaqAJEmSogpIkiQpqoAkSZKiCkiSJCmqgCRJkqIKSJIkKaqAJEmSogpIkiQpqoAkSZKiCkiSJCmqgCRJkqIKSJIkKaqAJEmSogpIkiQpqoAkSZKiCkiSJCmqgCRJkqIKSJIkKaqAJEmSogroG/GXv/yFZ599lsKmT59OUlISkiTp9BbQN+LVV1/loYceorAHH3yQlJQUJEnS6S0gSZKkqAKSJEmKKiBJkqSoApIkSYoqIEmSpKgCkiRJiiogSZKkqAKSJEmKKiBJkqSoApIkSYoqIEmSpKgCkiRJiiogSZKkqAKSJEmKKiBJkqSoApIkSYoqIEmSpKgCkiRJiiogSZKkqAKSJEmKKiBJkqSoApIkSYoqIEmSpKgCkiRJiiogSZKkqAKSJEmKKiBJkqSoApIkSYoqIEmSpKgCkiRJiiogSZKkqAKSJEmKKiBJkqSoApIkSYoqIEmSpKgCkiRJiiogSZKkqAKSJEmKKiBJkqSoApIkSYoqIEmSpKgCkiRJiiogSZKkqAKSJEmKKiBJkqSoApIkSYoqIEmSpKgCkiRJiiogSZKkqAKSJEmKKiBJkqSoApIkSYoqIEmSpKgCkiRJiiogSZKkqAKSJEmKKiBJkqSoApIkSYoqIEmSpKgCkiRJiiogSZKkqAKSJEmKKiBJkqSoApIkSYoqIEmSpKgCkiRJiiogSZKkqAKSJEmKKiBJkqSoApIkSYoqIEmSpKgCkiRJiiogSZKkqAKSJEmKKiBJkqSoApIkSYoqIEmSpKgCkiRJiiogSZKkqAKSJEmKKiBJkqSoApIkSYoqIEmSpKgCkiRJiiogSZKkqAKSJEmKKiBJkqSoApIkSYoqIEmSpKgCkiRJiiogSZKkqAKSJEmKKiBJkqSoApIkSYoqIEmSpKgCkiRJiiogSZKkqAKSJEmKKiBJkqSoApIkSYoqIEmSpKgCkiRJiiogSZKkqAKSJEmKKiBJkqSoApIkSYoqIEmSpKgCkiRJiiogSZKkqAKSJEmKKiBJkqSoApIkSYoqIEmSpKgCkiRJiiogSZKkqAKSJEmKKiBJkqSoApIkSYoqIEmSpKgCkiRJiiogSZKkqAKSJEmKKiBJkqSoApIkSYoqIEmSpKgCkiRJiiogSZKkqAKSJEmKKiBJkqSoApIkSYoqIEmSpKgCkiRJiiogSZKkqAKSJEmKKiBJkqSoApIkSYoqIEmSpKgCkiRJiiogSZKkqAKSJEmKKiBJkqSoApIkSYoqIEmSpKgCkiRJiiogSZKkqAKSJEmKKiBJkqSoAqeIX/3qV3zyyScUlZSUREZGBg0aNOCyyy7jW9/6FoUtW7aMZ555hs8zZMgQWrRogSRJ0tcVOEXMmjWL999/n5OpVasWixcvplmzZhR4/fXXmTZtGp+nU6dOtGjRAkmSpK8rcAopXbo0OTk5FHb06FHee+89pk6dyvz587n66qvZuHEjRd1zzz0MGjSIE0lPT0eSJCmGwCmmXLlyFFWpUiXmzJnDyy+/zKZNm9ixYwfnnXcehaWmplKuXDkkSZK+aYHTRAiBxo0bs337dnbu3Ml5553Hl7V161YWLVpEy5Ytufjiiyns97//PVu2bGHQoEFkZGQgSZJ0IoHTyPbt20lOTqZ27dp8Fd/+9rdZtGgRd955J5s2baJmzZokvP766/Tr149evXoxevRoJEmSTiZwGti/fz9Tpkxh8+bN/OQnP6Fq1aoUtXnzZpYsWcLxNG/enGrVqhFCIDs7m5YtWzJ48GCWLl3KkSNHGDRoEJUqVeKRRx7h69i7dy8pKSlIkqSvpmzZsiQnJ1PSBU4hhw4domrVqhR27NgxPv74Y/Lz8xk7diwTJkzgeB544AEeeOABjmfOnDn079+fhEaNGnHnnXfyi1/8gkceeYQPP/yQN998kz//+c9UqlSJL6phw4ZcdtllfPzxx6xevZqESpUqIUmSvro33niD5s2bU9IFTiHJycm0aNGCwvLz89m5cyebN29m3rx5DBo0iNq1a1PUoEGDuOiiizieDh06UNjPfvYz/vCHP/CLX/yCQ4cOMWzYMC655BK+jD59+tCnTx8++OADnnrqKSRJ0teXmZnJqSBwCilVqhRLlizheB599FF+/OMfc9tttzF//nyKatOmDf379+eLSE5O5uGHH6Zx48akp6dz991381VVr16dn/3sZ0iSpDNH4DQxZMgQbrnlFl599VVi+N3vfkfC/v37eeyxxxgxYgSSJElfROA0kpGRwUcffcTX9dJLL3Hvvfdy00038corrzBmzBguueQS6tatiyRJ0ucJnEby8vL4z3/+w9fx6aefMmDAAOrUqcOdd97Jjh07aNGiBQMHDmTlypUkJycjSZJ0MoHTyDnnnMPhw4fZunUr9erV46u44YYb+Ne//sXKlStJTU2lQYMGTJgwgdGjR3Pffffx85//HEmSpJMJnEYuuOAC1q9fz8iRI3nuuef4shYtWsScOXMYOXIkHTp0oMCoUaNYuHAhY8eOpWfPntSvXx9JkqQTCZwidu7cyeeZPn0606dPp7BRo0YxatQovogf/ehH5OfnU1RKSgqvvvoqkiRJX0RAkiRJUQUkSZIUVUCSJElRBSRJkhRVQJIkSVEFJEmSFFVAkiRJUQUkSZJKsP3793P11VdT2JAhQ+jZsyclVUCSJKkEO3z4MIsXL6awzp07U5IFJEmSFFVAkiRJUQUkSZIUVUCSJElRBSRJkhRVQJIkSVEFJEmSFFVAkiRJUQUkSZIUVUCSJElRBSRJkhRVQJIkSVEFJEmSFFVAkiRJUQUkSZIUVUCSJElRBSRJkhRVQJIkSVEFJEmSFFVAkiRJUQUkSZIUVUCSJElRBSRJkhRVQJIkSVEFJEmSFFVAkiRJUQUkSZIUVUCSJElRBSRJkhRVQJIkSVEFJEmSFFVAkiRJUQUkSZIUVUCSJElRBSRJkhRVQJIkSVEFJEmSFFVAkiRJUQUkSZIUVUCSJElRBSRJkhRVQJIkSVEFJEmSFFVAkiRJUQUkSZIUVUCSJElRBSRJkhRVQJIkSVEFJEmSFFVAkiRJUQUkSZIUVUCSJElRBSRJkhRVQJIkSVEFJEmSFFVAkiRJUQUkSZIUVUCSJElRBSRJkhRVQJIkSVEFJEmSFFVAkiRJUQUkSZIUVUCSJElRBSRJkhRVQJIkSVEFJEmSFFVAkiRJUQUkSZIUVUCSJElRBSRJkhRVQJIkSVEFJEmSFFVAkiRJUQUkSZIUVUCSJElRBSRJkhRVQJIkSVEFJEmSFFVAkiRJUQUkSZIUVUCSJElRBSRJkhRVQJIkSVEFJEmSFFVAkiRJUQUkSZIUVUCSJElRBSRJkhRVQJIkSVEFJEmSFFVAkiRJUQUkSZIUVUCSJElRBSRJkhRVQJIkSVEFJEmSFFVAkiRJUQUkSZIUVUCSJElRBSRJkhRVQJIkSVEFJEmSFFVAkiRJUQUkSZIUVUCSJElRBSRJkhRVQJIkSVEFJEmSFFVAkiRJUQUkSZI

## 2. Build a reusable axon substrate

We start with a random two-dimensional packing of axons in a periodic
box. Some axons are represented as cylinders and the others as
myelinated annuli. The same random positions and radii are reused in
every geometry variant so the fingerprint comparison reflects the chosen
change, not a new random draw.

In [1]:
box_size = 50.0  # μm
positions, radii = random_positions_radii(box_size, 0.6, 2; variance=0.0)
myelin_assignment = rand(length(positions))
degeneration_assignment = rand(length(positions))

function axon_geometry(
    ;
    scale=1.0,
    myelin_fraction=0.5,
    degeneration_fraction=0.0,
    orientation=:y,
    g_ratio=0.7,
)
    retained = degeneration_assignment .> degeneration_fraction
    cell_positions = positions[retained]
    cell_radii = radii[retained]
    is_myelinated = myelin_assignment[retained] .< myelin_fraction

    unmyelinated = Cylinders(
        radius=0.7 .* cell_radii[.!is_myelinated] .* scale,
        position=scale .* cell_positions[.!is_myelinated],
        density=1.0,
        dwell_time=800,
        R2_surface=10,
        repeats=(box_size * scale, box_size * scale),
        rotation=orientation,
        relaxation=5e-3,
    )
    myelinated = Annuli(
        inner=0.7 .* cell_radii[is_myelinated] .* scale,
        outer=(0.7 / g_ratio) .* cell_radii[is_myelinated] .* scale,
        position=scale .* cell_positions[is_myelinated],
        myelin=true,
        R2_inner_surface=10,
        R2_outer_surface=10,
        density_inner_surface=1.0,
        density_outer_surface=1.0,
        dwell_time_inner_surface=800,
        dwell_time_outer_surface=800,
        repeats=(box_size * scale, box_size * scale),
        rotation=orientation,
        relaxation_inner_surface=5e-3,
        relaxation_outer_surface=5e-3,
    )
    [unmyelinated, myelinated]
end

baseline_geometry = axon_geometry()
plot(baseline_geometry)

The baseline has a `0.6` target area fraction, with half of the axons
myelinated. `g_ratio` is the inner radius divided by the outer radius of
a myelinated axon. The defaults above give `g_ratio=0.7`.

## 3. Calculate the four fingerprint components

For each geometry, we read out the four non-MT sequences and one MT
sequence. The diffusion component is the b=1 signal divided by the b=0
signal. The other components are normalized by the estimated fraction of
free-water spins. Reduced sample counts make this example suitable for a
workshop run; the values will therefore have some Monte Carlo noise.

In [1]:
function multimodal_fingerprint(
    geometry;
    nspins=2_000,
    free_density_spins=10_000,
    size_scale=1.0,
)
    R1 = 1 / 2000
    R2 = 1 / 2000

    simulation = Simulation(
        no_mt_sequences;
        geometry=geometry,
        R1=R1,
        R2=R2,
        # The `tortuosity` parameter in the timestep sets the precision with which the extracellular diffusion should be simulated.
        # The default value of `0.03` produces accurate results, but is too slow for this workshop.
        # A value of `100` is used here to speed up the simulation, but it will produce a significant bias in the extracellular diffusion.
        timestep=(tortuosity=100., ),
    )
    no_mt_signal = transverse.(readout(nspins, simulation), mean=true)

    mt_simulation = Simulation(
        mt_sequence;
        geometry=geometry,
        R1=R1,
        R2=R2,
        timestep=(size_scale=Inf,),
        verbose=false,
    )
    mt_signal = transverse(readout(nspins, mt_simulation), mean=true)

    initial_snapshot = Snapshot(free_density_spins, mt_simulation)
    free_fraction = 1 - sum(MCMRSimulator.stuck.(initial_snapshot)) / free_density_spins

    [
        no_mt_signal[1] / no_mt_signal[2],
        no_mt_signal[3] / free_fraction,
        no_mt_signal[4] / free_fraction,
        mt_signal / free_fraction,
    ]
end

baseline_fingerprint = multimodal_fingerprint(baseline_geometry)
baseline_fingerprint

4-element Vector{Float64}:
 0.6869883072416098
 0.5363558625676388
 0.47295748547219496
 1.5264584898160625

The fingerprint values are ordered as: **DWI b=1/b=0**, **spin echo**,
**gradient echo**, and **magnetisation transfer**.

## 4. Change the substrate and compare fingerprints

We compare the baseline with four controlled changes: doubling axon
diameter, reducing the myelinated fraction, making the myelin thinner,
and removing 20% of the axons. Each change reuses the same sampled
positions and radii.

In [1]:
double_diameter = multimodal_fingerprint(
    axon_geometry(scale=2.0);
)
less_myelin = multimodal_fingerprint(axon_geometry(myelin_fraction=0.25))
thin_myelin = multimodal_fingerprint(axon_geometry(g_ratio=0.81))
axon_loss = multimodal_fingerprint(axon_geometry(degeneration_fraction=0.2))

variants = [
    ("Baseline", baseline_fingerprint),
    ("Diameter ×2", double_diameter),
    ("Myelin fraction 0.25", less_myelin),
    ("Thinner myelin", thin_myelin),
    ("20% axon loss", axon_loss),
]


5-element Vector{Tuple{String, Vector{Float64}}}:
 ("Baseline", [0.6869883072416098, 0.5363558625676388, 0.47295748547219496, 1.5264584898160625])
 ("Diameter ×2", [0.6218550913644922, 0.7147839568557327, 0.5917274386257411, 1.2686929543412588])
 ("Myelin fraction 0.25", [0.5287825584604348, 0.6512869817490808, 0.6176510176772939, 1.4426056064047679])
 ("Thinner myelin", [0.5073964530910452, 0.5987097419269788, 0.5824110847896277, 1.5171818318459154])
 ("20% axon loss", [0.5504751026879178, 0.6058910018876035, 0.5500961871036834, 1.4128527562697526])

Plot each variant relative to the baseline. Values above or below one
show how each MRI modality changes compared with the baseline substrate.

In [1]:
modality_labels = ["DWI", "SE", "GE", "MT"]
modality_colors = [:steelblue, :seagreen, :darkorange, :purple]
fingerprint_figure = Figure(size=(1100, 600))

for (index, (name, fingerprint)) in enumerate(variants)
    row = cld(index, 3)
    column = mod1(index, 3)
    axis = Axis(
        fingerprint_figure[row, column];
        title=name,
        xticks=(1:4, modality_labels),
        ylabel="Relative to baseline",
    )
    barplot!(axis, 1:4, fingerprint ./ baseline_fingerprint; color=modality_colors)
    hlines!(axis, [1.0]; color=:black, linestyle=:dash)
end
fingerprint_figure

Each modality responds differently to the substrate changes, which is
the central idea of a multimodal fingerprint: combining contrasts can
distinguish microstructural changes that may look similar in any single
measurement.

Cottaar, Michiel, Zhiyu Zheng, Karla Miller, Benjamin C. Tendler, and
Saad Jbabdi. 2026. “Multi-Modal Monte Carlo MRI Simulator of Tissue
Microstructure.” *Imaging Neuroscience* 4 (March): IMAG.a.1177.
<https://doi.org/10.1162/IMAG.a.1177>.